# DeepEval example

In [1]:
import google.generativeai as genai
import instructor
from pydantic import BaseModel

In [2]:
from langchain_google_vertexai import ChatVertexAI

llm = ChatVertexAI(model="gemini-1.5-pro", temperature=0, request_parallelism=1)

In [3]:
genai.configure(api_key="")

In [4]:
from deepeval.models import DeepEvalBaseLLM


class CustomGeminiFlash(DeepEvalBaseLLM):
    def __init__(self):
        self.model = genai.GenerativeModel(model_name="models/gemini-1.5-pro")

        # self.model = ChatVertexAI(model="gemini-1.5-pro", temperature=0, request_parallelism=1)

    def load_model(self):
        return self.model

    def generate(self, prompt: str, schema: BaseModel) -> BaseModel:
        client = self.load_model()
        instructor_client = instructor.from_gemini(
            client=client,
            mode=instructor.Mode.GEMINI_JSON,
        )
        resp = instructor_client.messages.create(
            messages=[
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            response_model=schema,
        )
        return resp

    async def a_generate(self, prompt: str, schema: BaseModel) -> BaseModel:
        return self.generate(prompt, schema)

    def get_model_name(self):
        return "Gemini 1.5 Pro"

In [5]:
custom_llm = CustomGeminiFlash()

In [ ]:
custom_llm = ChatVertexAI(model="gemini-1.5-pro", temperature=0, request_parallelism=1)

In [6]:
from deepeval.metrics import (
    AnswerRelevancyMetric,
    ContextualRelevancyMetric,
    HallucinationMetric,
)

answer_relevancy_metric = AnswerRelevancyMetric(model=custom_llm)
hallucination_metric = HallucinationMetric(threshold=0.5, model=custom_llm)
contextual_relevancy_metric = ContextualRelevancyMetric(threshold=0.5, model=custom_llm)

## read csv for the silver layer

In [7]:
!pwd

/home/baptvit/Documents/mestrado/master-experiments/evaluations


In [22]:
CONSUMER_ID = "Allen322_Ferry570_ad134528-56a5-35fd-c37f-466ff119c625"

In [23]:
import pandas as pd

df_silver = pd.read_csv(
    "/home/baptvit/Documents/mestrado/master-experiments/evaluations/data/silver/Allen322_Ferry570_ad134528-56a5-35fd-c37f-466ff119c625.csv"
)

In [24]:
df_silver.head()

,Unnamed: 0,experiment_id,timestamp_output_step,full_response,system_promt,input,output,timestamp_search_step,latency_s,query,...,total_time,timestamp_tool_step,records,caracteres_count,pass_map_reduce,resource_type,main_keys,user_query,llm_token_limit,strategy_name
0,0,Q1:gemini-1.5-pro:lexical_search_1_hop,2024-12-20 10:40:59,"{'input': ""What's my current medications and h...",\nAs an expert in interpreting Electronic Heal...,What's my current medications and how should I...,You are currently taking the following medicat...,2024-12-20 10:39:02,0.081823,MATCH (n)\n WHERE toLower(n.nam...,...,119.618273,2024-12-20 10:39:02,{'Main health record': 'Medication request sta...,363705,False,MedicationRequest,NaN,NaN,1000000,LexicalSearch1HopStrategy
1,1,Q2:gemini-1.5-pro:lexical_search_1_hop,2024-12-20 10:41:07,"{'input': 'What are my documented allergies, a...",\nAs an expert in interpreting Electronic Heal...,"What are my documented allergies, and how seve...",You have three documented allergies on file:\n...,2024-12-20 10:41:03,0.025798,MATCH (n)\n WHERE toLower(n.nam...,...,7.656272,2024-12-20 10:41:03,{'Main health record': 'Clinical Status: Activ...,1824,False,AllergyIntolerance,NaN,NaN,1000000,LexicalSearch1HopStrategy
2,2,Q3:gemini-1.5-pro:lexical_search_1_hop,2024-12-20 10:42:43,{'input': 'Can you summarize my current medica...,\nAs an expert in interpreting Electronic Heal...,Can you summarize my current medical conditions ?,This is a summary of your current medical cond...,2024-12-20 10:41:11,0.061930,MATCH (n)\n WHERE toLower(n.nam...,...,95.191523,2024-12-20 10:41:11,{'Main health record': 'Resource Type: Conditi...,516202,False,Condition,NaN,NaN,1000000,LexicalSearch1HopStrategy
3,3,Q4:gemini-1.5-pro:lexical_search_1_hop,2024-12-20 10:44:21,{'input': 'What are my recent laboratory value...,\nAs an expert in interpreting Electronic Heal...,"What are my recent laboratory values, what do ...",Please provide me with the specific laboratory...,2024-12-20 10:42:48,0.021954,MATCH (n)\n WHERE toLower(n.nam...,...,96.374906,2024-12-20 10:42:48,{'Main health record': 'Resource Type: Diagnos...,265054,False,DiagnosticReport,NaN,NaN,1000000,LexicalSearch1HopStrategy
4,4,Q5:gemini-1.5-pro:lexical_search_1_hop,2024-12-20 10:45:38,{'input': 'Can you summarize my care plan hist...,\nAs an expert in interpreting Electronic Heal...,Can you summarize my care plan history ?,This is a summary of your care plan history:\n...,2024-12-20 10:44:25,0.022743,MATCH (n)\n WHERE toLower(n.nam...,...,76.549560,2024-12-20 10:44:25,{'Main health record': 'Resource Type: CarePla...,19165,False,CarePlan,NaN,NaN,1000000,LexicalSearch1HopStrategy


In [25]:
df_silver.columns

Index(['Unnamed: 0', 'experiment_id', 'timestamp_output_step', 'full_response',
       'system_promt', 'input', 'output', 'timestamp_search_step', 'latency_s',
       'query', 'similarity_threshold', 'k', 'embedding_model',
       'timestamp_token_step', 'total_cost', 'total_tokens',
       'successful_requests', 'completion_tokens', 'prompt_tokens',
       'total_time', 'timestamp_tool_step', 'records', 'caracteres_count',
       'pass_map_reduce', 'resource_type', 'main_keys', 'user_query',
       'llm_token_limit', 'strategy_name'],
      dtype='object')

### Transforme the dataframe into LLMTest cases

In [26]:
import numpy as np
from deepeval.test_case import LLMTestCase

list_test_cases = []

for _, row in df_silver.iterrows():
    contexts = (
        "Not data available for this question."
        if row["records"] is np.nan
        else row["records"]
    )
    # print(contexts)
    test_case = LLMTestCase(
        name=row["experiment_id"],
        input=row["input"],
        actual_output=row["output"],
        retrieval_context=[contexts],
        context=[contexts],
    )
    list_test_cases.append(test_case)

In [27]:
from deepeval.dataset import EvaluationDataset

dataset = EvaluationDataset(test_cases=list_test_cases)

In [28]:
from deepeval import evaluate
from deepeval.dataset import EvaluationDataset

metrics = evaluate(
    dataset,
    [hallucination_metric, answer_relevancy_metric, contextual_relevancy_metric],
    ignore_errors=True,
    skip_on_missing_params=True,
)

✨ You're running DeepEval's latest Hallucination Metric! (using Gemini 1.5 Pro, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using Gemini 1.5 Pro, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using Gemini 1.5 Pro, strict=False, 
async_mode=True)...

Event loop is already running. Applying nest_asyncio patch to allow async execution...


Evaluating 32 test case(s) in parallel: |█████████████|100% (32/32) [Time Taken: 35:12, 66.01s/test case]
IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



✓ Tests finished 🎉! Run 'deepeval login' to save and analyze evaluation results on Confident AI. 
‼️  Friendly reminder 😇: You can also run evaluations with ALL of deepeval's metrics directly on Confident AI 
instead.

In [29]:
metrics_dict = metrics.dict()

In [30]:
rows = []
for result in metrics_dict["test_results"]:
    for metric in result["metrics_data"]:
        row = {
            "test_name": result["name"],
            "success": result["success"],
            "metric_name": metric["name"],
            "threshold": metric["threshold"],
            "metric_success": metric["success"],
            "score": metric["score"],
            "reason": metric["reason"],
            "strict_mode": metric["strict_mode"],
            "evaluation_model": metric["evaluation_model"],
            "error": metric["error"],
            "evaluation_cost": metric["evaluation_cost"],
            "verbose_logs": metric["verbose_logs"],
            "conversational": result["conversational"],
            "multimodal": result["multimodal"],
            "input": result["input"],
            "actual_output": result["actual_output"],
            "expected_output": result["expected_output"],
            "context": result["context"][0] if result["context"] else None,
            "retrieval_context": result["retrieval_context"][0]
            if result["retrieval_context"]
            else None,
        }
        rows.append(row)

In [31]:
# Create the DataFrame
df_gold = pd.DataFrame(rows)

# Display the DataFrame
df_gold["consumer_id"] = CONSUMER_ID
df_gold

,test_name,success,metric_name,threshold,metric_success,score,reason,strict_mode,evaluation_model,error,evaluation_cost,verbose_logs,conversational,multimodal,input,actual_output,expected_output,context,retrieval_context,consumer_id
0,Q1:gemini-1.5-pro:lexical_search_1_hop,True,Hallucination,0.5,True,0.0,The score is 0.00 because all information in t...,False,Gemini 1.5 Pro,None,None,"Verdicts:\n[\n {\n ""verdict"": ""yes"",...",False,False,What's my current medications and how should I...,You are currently taking the following medicat...,None,{'Main health record': 'Medication request sta...,{'Main health record': 'Medication request sta...,Allen322_Ferry570_ad134528-56a5-35fd-c37f-466f...
1,Q1:gemini-1.5-pro:lexical_search_1_hop,True,Answer Relevancy,0.5,True,1.0,The score is 1.00 because the response is a pe...,False,Gemini 1.5 Pro,None,None,"Statements:\n[\n ""You are currently taking ...",False,False,What's my current medications and how should I...,You are currently taking the following medicat...,None,{'Main health record': 'Medication request sta...,{'Main health record': 'Medication request sta...,Allen322_Ferry570_ad134528-56a5-35fd-c37f-466f...
2,Q1:gemini-1.5-pro:lexical_search_1_hop,True,Contextual Relevancy,0.5,True,1.0,The score is 1.00 because the retrieval contex...,False,Gemini 1.5 Pro,None,None,"Verdicts:\n[\n {\n ""verdicts"": [\n ...",False,False,What's my current medications and how should I...,You are currently taking the following medicat...,None,{'Main health record': 'Medication request sta...,{'Main health record': 'Medication request sta...,Allen322_Ferry570_ad134528-56a5-35fd-c37f-466f...
3,Q2:gemini-1.5-pro:lexical_search_1_hop,True,Hallucination,0.5,True,0.0,The score is 0.00 because the output accuratel...,False,Gemini 1.5 Pro,None,None,"Verdicts:\n[\n {\n ""verdict"": ""yes"",...",False,False,"What are my documented allergies, and how seve...",You have three documented allergies on file:\n...,None,{'Main health record': 'Clinical Status: Activ...,{'Main health record': 'Clinical Status: Activ...,Allen322_Ferry570_ad134528-56a5-35fd-c37f-466f...
4,Q2:gemini-1.5-pro:lexical_search_1_hop,True,Answer Relevancy,0.5,True,1.0,The score is 1.00 because the response is a pe...,False,Gemini 1.5 Pro,None,None,"Statements:\n[\n ""You have three documented...",False,False,"What are my documented allergies, and how seve...",You have three documented allergies on file:\n...,None,{'Main health record': 'Clinical Status: Activ...,{'Main health record': 'Clinical Status: Activ...,Allen322_Ferry570_ad134528-56a5-35fd-c37f-466f...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,Q7:gemini-1.5-pro:similarity_search_1_hop,True,Answer Relevancy,0.5,True,1.0,The score is 1.00 because the response is a pe...,False,Gemini 1.5 Pro,None,None,"Statements:\n[\n ""This is a summary of the ...",False,False,"What procedures have I undergone recently, and...",This is a summary of the procedures you have u...,None,{'Main health record': 'Procedure Status: Comp...,{'Main health record': 'Procedure Status: Comp...,Allen322_Ferry570_ad134528-56a5-35fd-c37f-466f...
92,Q7:gemini-1.5-pro:similarity_search_1_hop,True,Contextual Relevancy,0.5,True,1.0,The score is 1.00 because the retrieval contex...,False,Gemini 1.5 Pro,None,None,"Verdicts:\n[\n {\n ""verdicts"": [\n ...",False,False,"What procedures have I undergone recently, and...",This is a summary of the procedures you have u...,None,{'Main health record': 'Procedure Status: Comp...,{'Main health record': 'Procedure Status: Comp...,Allen322_Ferry570_ad134528-56a5-35fd-c37f-466f...
93,Q8:gemini-1.5-pro:similarity_search_1_hop,True,Hallucination,0.5,True,0.0,The score is 0.00 because the output accuratel...,False,Gemini 1.5 Pro,None,None,"Verdicts:\n[\n {\n ""verdict"": ""yes"",...",False,False,Can you summarize my immunization history ?,You received the seasonal influenza vaccine on...,None,{'Main health record': 'Immunization Status: c...,{'Ma

In [32]:
df_gold.columns

Index(['test_name', 'success', 'metric_name', 'threshold', 'metric_success',
       'score', 'reason', 'strict_mode', 'evaluation_model', 'error',
       'evaluation_cost', 'verbose_logs', 'conversational', 'multimodal',
       'input', 'actual_output', 'expected_output', 'context',
       'retrieval_context', 'consumer_id'],
      dtype='object')

In [33]:
df_final = df_gold[
    [
        "consumer_id",
        "test_name",
        "success",
        "metric_name",
        "threshold",
        "metric_success",
        "score",
        "reason",
        "strict_mode",
        "evaluation_model",
        "error",
        "evaluation_cost",
        "verbose_logs",
        "conversational",
        "multimodal",
        "input",
        "actual_output",
        "expected_output",
        "context",
        "retrieval_context",
    ]
]

In [34]:
df_final.to_csv(
    "/home/baptvit/Documents/mestrado/master-experiments/evaluations/data/gold/Allen322_Ferry570_ad134528-56a5-35fd-c37f-466ff119c625_deepeval.csv"
)

## Work with datasets

In [ ]:
from deepeval.dataset import EvaluationDataset

In [ ]:
dataset = EvaluationDataset(test_cases=[test_case])

In [ ]:
from deepeval import evaluate
from deepeval.dataset import EvaluationDataset

hallucination_metric = HallucinationMetric(threshold=0.3, model=custom_llm)

metrics = evaluate(dataset, [hallucination_metric, metric])

In [ ]:
test_json = test.json()

In [ ]:
test_dict = test.dict()

In [ ]:
test_dict

In [ ]:
test_dict

In [ ]:
test_dict["test_results"][0]["metrics_data"][0]

In [ ]:
import pandas as pd

data = {
    "test_results": [
        {
            "name": "test_case_0",
            "success": True,
            "metrics_data": [
                {
                    "name": "Hallucination",
                    "threshold": 0.3,
                    "success": True,
                    "score": 0.0,
                    "reason": "The score is 0.00 because the actual output perfectly aligns with the provided context, confirming the 30-day full refund policy without introducing any contradictory or hallucinated information.",
                    "strict_mode": False,
                    "evaluation_model": "Gemini 1.5 Pro",
                    "error": None,
                    "evaluation_cost": None,
                    "verbose_logs": 'Verdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": "The actual output agrees with the provided context. The context mentions that all customers are eligible for a 30-day full refund at no extra cost. The actual output confirms the 30-day full refund policy."\n    }\n]',
                },
                {
                    "name": "Answer Relevancy",
                    "threshold": 0.5,
                    "success": True,
                    "score": 1.0,
                    "reason": "The score is 1.00 because the response is a perfect and concise answer to the question.  It directly addresses the user's concern about the shoes not fitting. There are no irrelevant statements, so the score cannot be any higher. Keep up the great work!",
                    "strict_mode": False,
                    "evaluation_model": "Gemini 1.5 Pro",
                    "error": None,
                    "evaluation_cost": None,
                    "verbose_logs": 'Statements:\n[\n    "We offer a 30-day full refund at no extra cost."\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    }\n]',
                },
            ],
            "conversational": False,
            "multimodal": False,
            "input": "What if these shoes don't fit?",
            "actual_output": "We offer a 30-day full refund at no extra cost.",
            "expected_output": None,
            "context": [
                "All customers are eligible for a 30 day full refund at no extra cost."
            ],
            "retrieval_context": [
                "All customers are eligible for a 30 day full refund at no extra cost."
            ],
        }
    ],
    "confident_link": None,
}

In [ ]:
def dict_to_dataframe(data):
    """Transforms the nested dictionary into a pandas DataFrame."""

    test_results = data.get("test_results", [])
    rows = []

    for test_case in test_results:
        metrics_data = test_case.pop("metrics_data", [])  # Extract metrics data
        base_data = test_case  # all the other data
        for metric in metrics_data:
            row = base_data.copy()  # Create a copy to avoid modifying the original
            row.update(metric)  # Merge metric data into the base data
            rows.append(row)

    df = pd.DataFrame(rows)
    return df


df = dict_to_dataframe(data)
df

In [ ]:
# If you want a cleaner output without verbose logs and reasons:
def dict_to_dataframe_clean(data):
    """Transforms the nested dictionary into a pandas DataFrame, removing verbose logs and reasons."""

    df = dict_to_dataframe(data)
    columns_to_drop = ["verbose_logs", "reason"]
    df = df.drop(
        columns=columns_to_drop, errors="ignore"
    )  # errors='ignore' handles cases where the column is not present
    return df


df_clean = dict_to_dataframe_clean(data)
print("\nCleaned DataFrame:")
df_clean

In [ ]:
# If you have multiple test cases in test_results:
data_multiple = {
    "test_results": [
        {
            "name": "test_case_0",
            "success": True,
            "metrics_data": [
                {
                    "name": "Hallucination",
                    "threshold": 0.3,
                    "success": True,
                    "score": 0.0,
                },
                {
                    "name": "Answer Relevancy",
                    "threshold": 0.5,
                    "success": True,
                    "score": 1.0,
                },
            ],
        },
        {
            "name": "test_case_1",
            "success": False,
            "metrics_data": [
                {
                    "name": "Hallucination",
                    "threshold": 0.3,
                    "success": False,
                    "score": 1.0,
                }
            ],
        },
    ],
    "confident_link": None,
}
df_multiple = dict_to_dataframe(data_multiple)
print("\nMultiple Test Cases DataFrame:")
print(df_multiple)

In [ ]:
df_multiple_clean = dict_to_dataframe_clean(data_multiple)
print("\nMultiple Test Cases Cleaned DataFrame:")
print(df_multiple_clean)

In [ ]:
# If you want to flatten the context and retrieval context lists:
def dict_to_dataframe_flatten_context(data):
    df = dict_to_dataframe(data)
    df["context"] = df["context"].apply(
        lambda x: ", ".join(x) if isinstance(x, list) else x
    )
    df["retrieval_context"] = df["retrieval_context"].apply(
        lambda x: ", ".join(x) if isinstance(x, list) else x
    )
    return df


df_flattened = dict_to_dataframe_flatten_context(data)
print("\nDataFrame with flattened context:")
print(df_flattened)

In [5]:
import pandas as pd

In [6]:
df_gold_Test = pd.read_csv("/home/baptvit/repositories/graphrag-on-fhir/evaluations/data/gold/Beatris270_Bogan287_5b3645de-a2d0-d016-0839-bab3757c4c58-gpt-4o-2024-08-06_deepeval.csv")

In [7]:
df_gold_Test

,consumer_id,test_name,success,metric_name,threshold,metric_success,score,reason,strict_mode,evaluation_model,error,evaluation_cost,verbose_logs,conversational,multimodal,input,actual_output,expected_output,context,retrieval_context
0,Beatris270_Bogan287_5b3645de-a2d0-d016-0839-ba...,Q1:gpt-4o-2024-08-06:similarity_search_0_hop,False,Hallucination,0.5,True,0.000000,The score is 0.00 because there are no contrad...,False,Custom Azure OpenAI Model,NaN,NaN,"Verdicts:\n[\n {\n ""verdict"": ""yes"",...",False,False,What's my current medications and how should I...,"Based on your electronic health records, here ...",NaN,{'Main health record': 'Resource Type: Medicat...,{'Main health record': 'Resource Type: Medicat...
1,Beatris270_Bogan287_5b3645de-a2d0-d016-0839-ba...,Q1:gpt-4o-2024-08-06:similarity_search_0_hop,False,Answer Relevancy,0.5,False,0.240000,The score is 0.24 because the output contains ...,False,Custom Azure OpenAI Model,NaN,NaN,"Statements:\n[\n ""Based on your electronic ...",False,False,What's my current medications and how should I...,"Based on your electronic health records, here ...",NaN,{'Main health record': 'Resource Type: Medicat...,{'Main health record': 'Resource Type: Medicat...
2,Beatris270_Bogan287_5b3645de-a2d0-d016-0839-ba...,Q1:gpt-4o-2024-08-06:similarity_search_0_hop,False,Contextual Relevancy,0.5,False,0.454545,The score is 0.45 because while some statement...,False,Custom Azure OpenAI Model,NaN,NaN,"Verdicts:\n[\n {\n ""verdicts"": [\n ...",False,False,What's my current medications and how should I...,"Based on your electronic health records, here ...",NaN,{'Main health record': 'Resource Type: Medicat...,{'Main health record': 'Resource Type: Medicat...
3,Beatris270_Bogan287_5b3645de-a2d0-d016-0839-ba...,Q8:gpt-4o-2024-08-06:similarity_search_0_hop,True,Hallucination,0.5,True,0.000000,The score is 0.00 because the actual output fu...,False,Custom Azure OpenAI Model,NaN,NaN,"Verdicts:\n[\n {\n ""verdict"": ""yes"",...",False,False,Can you summarize my immunization history ?,Here is a summary of your immunization history...,NaN,{'Main health record': 'Status: Completed\nVac...,{'Main health record': 'Status: Completed\nVac...
4,Beatris270_Bogan287_5b3645de-a2d0-d016-0839-ba...,Q8:gpt-4o-2024-08-06:similarity_search_0_hop,True,Answer Relevancy,0.5,True,1.000000,The score is 1.00 because the output is fully ...,False,Custom Azure OpenAI Model,NaN,NaN,"Statements:\n[\n ""Here is a summary of your...",False,False,Can you summarize my immunization history ?,Here is a summary of your immunization history...,NaN,{'Main health record': 'Status: Completed\nVac...,{'Main health record': 'Status: Completed\nVac...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,Beatris270_Bogan287_5b3645de-a2d0-d016-0839-ba...,Q7:gpt-4o-2024-08-06:similarity_search_1_hop,False,Answer Relevancy,0.5,True,1.000000,The score is 1.00 because the output is fully ...,False,Custom Azure OpenAI Model,NaN,NaN,"Statements:\n[\n ""Based on your recent medi...",False,False,"What procedures have I undergone recently, and...","Based on your recent medical records, here are...",NaN,{'Main health record': 'Procedure Status: Comp...,{'Main health record': 'Procedure Status: Comp...
92,Beatris270_Bogan287_5b3645de-a2d0-d016-0839-ba...,Q7:gpt-4o-2024-08-06:similarity_search_1_hop,False,Contextual Relevancy,0.5,True,0.600000,The score is 0.60 because while there are some...,False,Custom Azure OpenAI Model,NaN,NaN,"Verdicts:\n[\n {\n ""verdicts"": [\n ...",False,False,"What procedures have I undergone recently, and...","Based on your recent medical records, here are...",NaN,{'Main health record': 'Procedure Status: Comp...,{'Main health record': 'Procedure Status: Comp...
93,Beatris270_Bogan287_5b3645de-a2d0-d016-0839-ba...,Q3:gpt-4o-2024-08-06:similarity_search_1_hop,True,Hallucination,0.5,True,0.000000,The score is 0.00 because the actual output fu...,False,Custom Azure OpenAI Model,NaN,NaN,"Verdicts:\n[\n 

# Transform in pandas dataframe

In [ ]:
import pandas as pd

# The given dictionary
data = {
    "test_results": [
        {
            "name": "test_case_0",
            "success": True,
            "metrics_data": [
                {
                    "name": "Hallucination",
                    "threshold": 0.3,
                    "success": True,
                    "score": 0.0,
                    "reason": "The score is 0.00 because the actual output perfectly aligns with the provided context, confirming the 30-day full refund policy without introducing any contradictory or hallucinated information.",
                    "strict_mode": False,
                    "evaluation_model": "Gemini 1.5 Pro",
                    "error": None,
                    "evaluation_cost": None,
                    "verbose_logs": 'Verdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": "The actual output agrees with the provided context. The context mentions that all customers are eligible for a 30-day full refund at no extra cost. The actual output confirms the 30-day full refund policy."\n    }\n]',
                },
                {
                    "name": "Answer Relevancy",
                    "threshold": 0.5,
                    "success": True,
                    "score": 1.0,
                    "reason": "The score is 1.00 because the response is a perfect and concise answer to the question.  It directly addresses the user's concern about the shoes not fitting. There are no irrelevant statements, so the score cannot be any higher. Keep up the great work!",
                    "strict_mode": False,
                    "evaluation_model": "Gemini 1.5 Pro",
                    "error": None,
                    "evaluation_cost": None,
                    "verbose_logs": 'Statements:\n[\n    "We offer a 30-day full refund at no extra cost."\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    }\n]',
                },
            ],
            "conversational": False,
            "multimodal": False,
            "input": "What if these shoes don't fit?",
            "actual_output": "We offer a 30-day full refund at no extra cost.",
            "expected_output": None,
            "context": [
                "All customers are eligible for a 30 day full refund at no extra cost."
            ],
            "retrieval_context": [
                "All customers are eligible for a 30 day full refund at no extra cost."
            ],
        }
    ],
    "confident_link": None,
}

# Transforming the nested dictionary into a DataFrame
rows = []
for result in data["test_results"]:
    for metric in result["metrics_data"]:
        row = {
            "test_name": result["name"],
            "success": result["success"],
            "metric_name": metric["name"],
            "threshold": metric["threshold"],
            "metric_success": metric["success"],
            "score": metric["score"],
            "reason": metric["reason"],
            "strict_mode": metric["strict_mode"],
            "evaluation_model": metric["evaluation_model"],
            "error": metric["error"],
            "evaluation_cost": metric["evaluation_cost"],
            "verbose_logs": metric["verbose_logs"],
            "conversational": result["conversational"],
            "multimodal": result["multimodal"],
            "input": result["input"],
            "actual_output": result["actual_output"],
            "expected_output": result["expected_output"],
            "context": result["context"][0] if result["context"] else None,
            "retrieval_context": result["retrieval_context"][0]
            if result["retrieval_context"]
            else None,
        }
        rows.append(row)

In [ ]:
# Create the DataFrame
df = pd.DataFrame(rows)

# Display the DataFrame
df